In [8]:
# notebooks/02_data_validation.ipynb
import sys
import os
import pandas as pd
import numpy as np

# 1. Framework Path Setup
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_validation_splits
from src.metrics import calculate_psi
from src.scorecard_transformer import ScorecardTransformer

print("=== RUNNING PHASE 2: ADVANCED MRM DATA INTEGRITY & DRIFT AUDIT ===")

# 2. Ingest Data Splits from Disk
df_train_tweaked, df_val_final, df_oot_final, df_scorecard_points = load_validation_splits()

# Initialize our Phase 0 reusable software transformer layer
transformer = ScorecardTransformer(df_scorecard_points)

# =====================================================================
# PART A: YOUR ORIGINAL DATA COMPLIANCE & LEAKAGE AUDIT (PRESERVED)
# =====================================================================
print("\nExecuting Data Compliance & Leakage Isolation Protocols...")

def verify_target_definition(df):
    """Ensures intermediate risk accounts did not pollute the binary target."""
    leaked_statuses = ['In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)']
    if 'loan_status' in df.columns:
        polluted_rows = df[df['loan_status'].isin(leaked_statuses)]
        assert len(polluted_rows) == 0, f"FAIL: Found {len(polluted_rows)} indeterminate rows polluting target!"
        print("✅ Target Definition Pass: No indeterminate credit accounts found.")
    else:
        print("ℹ️ Target Definition Note: 'loan_status' raw column omitted; verified upstream.")

def verify_leakage_exclusion(df_columns):
    """Ensures post-origination behavior metrics are excluded."""
    leaked_features = ["total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_amnt"]
    detected_leaks = [feat for feat in leaked_features if feat in df_columns]
    assert len(detected_leaks) == 0, f"FAIL: Data Leakage Detected! Features present: {detected_leaks}"
    print("✅ Post-Origination Pass: Zero behavior-leakage variables detected.")

verify_target_definition(df_train_tweaked)
verify_leakage_exclusion(df_train_tweaked.columns)

# =====================================================================
# PART B: THE MISSING VALUE DRIFT AUDIT (MRM REQUIREMENT)
# =====================================================================
print("\nExecuting Production Missingness Integrity Scan...")

target_features = ['dti', 'annual_inc', 'fico_score', 'loan_amnt', 'grade']
missing_audit = []

for var in target_features:
    train_miss = df_train_tweaked[var].isna().mean() * 100
    oot_miss = df_oot_final[var].isna().mean() * 100
    missing_audit.append({
        "Variable": var,
        "Train Missing (%)": round(train_miss, 4),
        "OOT Missing (%)": round(oot_miss, 4),
        "Difference": round(oot_miss - train_miss, 4)
    })

print(pd.DataFrame(missing_audit).to_string(index=False))

# =====================================================================
# PART C: GLOBAL POPULATION STABILITY (SCORE PSI)
# =====================================================================
print("\nGenerating Production Credit Scores via Transformer Asset...")
df_train_tweaked['credit_score'] = transformer.generate_credit_scores(df_train_tweaked)
df_oot_final['credit_score'] = transformer.generate_credit_scores(df_oot_final)

real_psi, psi_table = calculate_psi(df_train_tweaked['credit_score'], df_oot_final['credit_score'], num_bins=10)

print("\n=== THE REAL MODEL GOVERNANCE SIGN-OFF ===")
print(f"True Population Stability Index (PSI): {real_psi:.4f}")
if real_psi < 0.10:
    print("✅ STATUS: Population is highly stable. Approved for production baseline.")
else:
    print("⚠️ STATUS: Significant population drift detected.")

# =====================================================================
# PART D: CHARACTERISTIC STABILITY INDEX / VARIABLE PSI (MRM REQUIREMENT)
# =====================================================================
print("\nCalculating Variable-Level Characteristic Stability Index (CSI)...")

def calculate_continuous_variable_psi(series_train, series_oot, bins=5):
    """Dynamically cuts continuous attributes into historical quintiles to isolate feature drift."""
    counts, boundaries = pd.qcut(series_train, q=bins, retbins=True, labels=False, duplicates='drop')
    boundaries[0] = float('-inf')
    boundaries[-1] = float('inf')
    
    train_counts = pd.cut(series_train, bins=boundaries).value_counts(sort=False)
    oot_counts = pd.cut(series_oot, bins=boundaries).value_counts(sort=False)
    
    train_prop = (train_counts / len(series_train)).values
    oot_prop = (oot_counts / len(series_oot)).values
    
    train_prop = np.where(train_prop == 0, 0.0001, train_prop)
    oot_prop = np.where(oot_prop == 0, 0.0001, oot_prop)
    
    return np.sum((oot_prop - train_prop) * np.log(oot_prop / train_prop))

csi_ledger = []
for var in ['dti', 'annual_inc', 'fico_score', 'loan_amnt']:
    v_psi = calculate_continuous_variable_psi(df_train_tweaked[var].dropna(), df_oot_final[var].dropna())
    csi_ledger.append({"Variable": var, "Stability Index (CSI)": round(v_psi, 5)})

for var in ['grade']:
    t_prop = df_train_tweaked[var].value_counts(normalize=True).to_dict()
    o_prop = df_oot_final[var].value_counts(normalize=True).to_dict()
    all_cats = set(t_prop.keys()).union(set(o_prop.keys()))
    psi_val = 0
    for cat in all_cats:
        p_t = t_prop.get(cat, 0.0001)
        p_o = o_prop.get(cat, 0.0001)
        psi_val += (p_o - p_t) * np.log(p_o / p_t)
    csi_ledger.append({"Variable": var, "Stability Index (CSI)": round(psi_val, 5)})

df_csi_report = pd.DataFrame(csi_ledger)

def assign_mrm_status(val):
    if val < 0.10: return "Stable (Green)"
    elif val < 0.25: return "Action Trigger Warning (Yellow)"
    return "Action Mandatory Drift (Red)"

df_csi_report['Action Threshold Status'] = df_csi_report['Stability Index (CSI)'].apply(assign_mrm_status)
print(df_csi_report.to_string(index=False))

=== RUNNING PHASE 2: ADVANCED MRM DATA INTEGRITY & DRIFT AUDIT ===

Executing Data Compliance & Leakage Isolation Protocols...
ℹ️ Target Definition Note: 'loan_status' raw column omitted; verified upstream.
✅ Post-Origination Pass: Zero behavior-leakage variables detected.

Executing Production Missingness Integrity Scan...
  Variable  Train Missing (%)  OOT Missing (%)  Difference
       dti                0.0              0.0         0.0
annual_inc                0.0              0.0         0.0
fico_score                0.0              0.0         0.0
 loan_amnt                0.0              0.0         0.0
     grade                0.0              0.0         0.0

Generating Production Credit Scores via Transformer Asset...

=== THE REAL MODEL GOVERNANCE SIGN-OFF ===
True Population Stability Index (PSI): 0.0071
✅ STATUS: Population is highly stable. Approved for production baseline.

Calculating Variable-Level Characteristic Stability Index (CSI)...
  Variable  Stability Index